# Module 4 — Quantization, Context, and Memory Math

Companion notebook to [`docs/modules/04_quantization_context_and_memory_math.md`](../docs/modules/04_quantization_context_and_memory_math.md).

This notebook:

1. Reproduces every worked example from the theory doc as real computed numbers (proving
   `memory_math.py` matches the curriculum exactly).
2. Demonstrates the memory-sampling tooling (`memory_sampler.py`) against a **dummy
   subprocess** — real, working measurement code, proven without needing a model runtime.
3. Runs the predict-then-measure lab against real Ollama models, if available (expected to
   skip on this machine — see `PROGRESS.md`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_04"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Weights: the quantization table, reproduced

Theory doc §1 rule-of-thumb table for an 8B model, in decimal GB (see `memory_math.py`'s
unit-note docstring for why weights use decimal GB while KV cache uses binary GiB — that's
the curriculum bible's own convention, matched exactly rather than silently "fixed").

In [2]:
from memory_math import QUANT_BITS_PER_PARAM, bytes_to_gb, weights_bytes

n_params_8b = 8_000_000_000
print(f"{'Quant':10s} {'GB (decimal)':>14s}")
for quant in QUANT_BITS_PER_PARAM:
    gb = bytes_to_gb(weights_bytes(n_params_8b, quant))
    print(f"{quant:10s} {gb:14.2f}")

Quant        GB (decimal)
FP16                16.00
Q8_0                 8.50
Q6_K                 6.60
Q5_K_M               5.70
Q4_K_M               4.80
Q3_K_M               3.90
Q2_K                 3.40


## 2. KV cache: the worked example and context table, reproduced

Theory doc §4: 8B Llama-style model, `n_layers=32`, `n_kv_heads=8`, `head_dim=128`.

In [3]:
from memory_math import bytes_to_gib, kv_cache_bytes

per_token_kib = kv_cache_bytes(n_layers=32, n_kv_heads=8, head_dim=128, context_tokens=1) / 1024
print(f"Per-token KV cache at FP16: {per_token_kib:.0f} KiB/token (theory doc says 128 KiB/token)\n")

print(f"{'Context':>10s} {'FP16 (GiB)':>12s} {'Q8 (GiB)':>10s} {'Q4 (GiB)':>10s}")
for ctx in [4_096, 8_192, 32_768, 128_000]:
    fp16 = bytes_to_gib(kv_cache_bytes(32, 8, 128, ctx, kv_quant="FP16"))
    q8 = bytes_to_gib(kv_cache_bytes(32, 8, 128, ctx, kv_quant="Q8_0"))
    q4 = bytes_to_gib(kv_cache_bytes(32, 8, 128, ctx, kv_quant="Q4_0"))
    print(f"{ctx:>10,} {fp16:>12.3f} {q8:>10.3f} {q4:>10.3f}")

Per-token KV cache at FP16: 128 KiB/token (theory doc says 128 KiB/token)

   Context   FP16 (GiB)   Q8 (GiB)   Q4 (GiB)
     4,096        0.500      0.250      0.125
     8,192        1.000      0.500      0.250
    32,768        4.000      2.000      1.000
   128,000       15.625      7.812      3.906


## 3. Concurrency multiplies the KV-cache term directly

In [4]:
print(f"{'Concurrent':>10s} {'KV cache @ 8K, FP16 (GiB)':>28s}")
for n in [1, 2, 4, 8]:
    gib = bytes_to_gib(kv_cache_bytes(32, 8, 128, 8_192, concurrent_sequences=n))
    print(f"{n:>10} {gib:>28.3f}")

Concurrent    KV cache @ 8K, FP16 (GiB)
         1                        1.000
         2                        2.000
         4                        4.000
         8                        8.000


## 4. Full budget, using the model-shape registry

`model_shapes.py` holds documented (not measured) architecture shapes for a few course
models. `estimate_memory_budget` combines weights + KV cache + a planning-grade runtime
overhead range — this is the "predicted" half of Lab 4.4's prediction-vs-actual table.

In [5]:
from memory_math import estimate_memory_budget
from model_shapes import KNOWN_SHAPES, get_shape

for model_id in KNOWN_SHAPES:
    shape = get_shape(model_id)
    print(f"## {model_id}  ({shape.source_note})")
    for ctx in [2_000, 8_000, 16_000]:
        est = estimate_memory_budget(
            n_params=shape.n_params, quant="Q4_K_M", n_layers=shape.n_layers,
            n_kv_heads=shape.n_kv_heads, head_dim=shape.head_dim, context_tokens=ctx,
        )
        print(f"  context={ctx:>6,}  weights={est.weights_gb:.2f}GB  kv={est.kv_cache_gib:.2f}GiB  "
              f"total={est.total_low_gib:.2f}-{est.total_high_gib:.2f}")
    print()

## llama3.1-8b  (Llama 3.1 8B published config (GQA, 8 KV heads) — matches theory doc's worked example)
  context= 2,000  weights=4.80GB  kv=0.24GiB  total=5.54-6.54
  context= 8,000  weights=4.80GB  kv=0.98GiB  total=6.28-7.28
  context=16,000  weights=4.80GB  kv=1.95GiB  total=7.25-8.25

## qwen2.5-7b  (Qwen2.5-7B published config (GQA, 4 KV heads))
  context= 2,000  weights=4.56GB  kv=0.11GiB  total=5.17-6.17
  context= 8,000  weights=4.56GB  kv=0.43GiB  total=5.49-6.49
  context=16,000  weights=4.56GB  kv=0.85GiB  total=5.91-6.91

## qwen2.5-1.5b  (Qwen2.5-1.5B published config (GQA, 2 KV heads))
  context= 2,000  weights=0.90GB  kv=0.05GiB  total=1.45-2.45
  context= 8,000  weights=0.90GB  kv=0.21GiB  total=1.61-2.61
  context=16,000  weights=0.90GB  kv=0.43GiB  total=1.83-2.83

## qwen2.5-coder-7b  (Qwen2.5-Coder-7B shares the Qwen2.5-7B base architecture)
  context= 2,000  weights=4.56GB  kv=0.11GiB  total=5.17-6.17
  context= 8,000  weights=4.56GB  kv=0.43GiB  total=5.49-6.49
 

## 5. Prove the memory sampler actually works (dummy subprocess, no model needed)

`memory_sampler.py` is real, working tooling, not a stub. We can't point it at a model
runtime here (none installed - see the machine constraint), but we can prove it correctly
tracks a peak RSS by pointing it at a dummy subprocess that deliberately allocates and holds
memory for a moment.

In [6]:
import subprocess

from memory_sampler import PeakMemorySampler

# A dummy process that allocates ~200MB, holds it briefly, then exits - so we can watch
# the sampler's tracked peak actually rise and then the process disappear.
dummy_code = (
    "import time; "
    "data = bytearray(200 * 1024 * 1024); "
    "time.sleep(0.6)"
)
proc = subprocess.Popen([sys.executable, "-c", dummy_code])

sampler = PeakMemorySampler(proc.pid, poll_interval_seconds=0.02)
sampler.start()
proc.wait()
peak = sampler.stop()

print(f"Dummy process pid: {proc.pid}")
print(f"Tracked peak RSS: {peak / 1024**2:.1f} MiB" if peak else "No sample captured")
assert peak is not None and peak > 100 * 1024 * 1024, "expected the sampler to see the ~200MB allocation"
print("\nConfirmed: the sampler correctly tracked a peak above the ~200MB the dummy process allocated.")

Dummy process pid: 17537
Tracked peak RSS: 209.1 MiB

Confirmed: the sampler correctly tracked a peak above the ~200MB the dummy process allocated.


## 6. Run the real labs against Ollama, if available

Expected to skip on this machine (no local model runtime installed by design). On a
resourced Mac, this becomes a real prediction-vs-actual measurement.

In [7]:
from ollama_probe import is_ollama_available
import lab_4_4_predict_then_measure as lab44

if is_ollama_available():
    rows = lab44.predict_and_measure(
        model_tag="qwen2.5:7b-instruct-q4_K_M", shape_id="qwen2.5-7b", quant="Q4_K_M",
        context_lengths=[2_000, 8_000, 16_000],
    )
    from IPython.display import Markdown, display
    display(Markdown(lab44.rows_to_markdown_table(rows)))
else:
    print("SKIPPED: Ollama is not reachable at http://localhost:11434.")
    print("This machine deliberately has no local model runtime installed (see PROGRESS.md).")
    print("On a resourced Mac: pull qwen2.5:7b-instruct-q4_K_M, then re-run this cell.")

SKIPPED: Ollama is not reachable at http://localhost:11434.
This machine deliberately has no local model runtime installed (see PROGRESS.md).
On a resourced Mac: pull qwen2.5:7b-instruct-q4_K_M, then re-run this cell.


## 7. Take it to the command line

```bash
uv run python scripts/module_04/lab_4_1_quantization_comparison.py --tags qwen2.5:7b-instruct-q4_K_M qwen2.5:7b-instruct-q8_0
uv run python scripts/module_04/lab_4_2_context_scaling.py --model qwen2.5:3b
uv run python scripts/module_04/lab_4_3_concurrency_simulation.py --model qwen2.5:3b
uv run python scripts/module_04/lab_4_4_predict_then_measure.py --model-tag qwen2.5:7b-instruct-q4_K_M --shape qwen2.5-7b
```

Then fold the output into
[`reports/module_04_quantization_context_memory_report.md`](../reports/module_04_quantization_context_memory_report.md).